# 99 — Colab fallback (DLC SuperAnimal-Quadruped)

**When to use:** local DLC/torch install failed on macOS Apple Silicon. Open this file directly in [Google Colab](https://colab.research.google.com/) (File → Upload notebook → pick `99_colab_fallback.ipynb`). Free-tier T4 is sufficient.

**This notebook does NOT require anything from poc/.venv or checkpoints/.** Everything is fetched fresh from PyPI / HuggingFace.

**Goal:** complete items 1 and 2 from GATE.md (setup + sensible keypoints) in a Google-managed environment.


## 1. Install (~5 minutes)

In [ ]:
!pip install --quiet "deeplabcut[modelzoo]>=3.0.0" "huggingface_hub>=0.24" "opencv-python-headless>=4.10"

In [ ]:
import torch, deeplabcut, sys
print('Python :', sys.version.split()[0])
print('Torch  :', torch.__version__, '| CUDA dostepne:', torch.cuda.is_available())
print('DLC    :', deeplabcut.__version__)
if torch.cuda.is_available():
    print('GPU    :', torch.cuda.get_device_name(0))

## 2. Get a sample horse video

Option A: upload your own video via the Files panel (left column). Option B: pull a fragment from a YouTube CC-BY video via yt-dlp.


In [ ]:
!pip install --quiet "yt-dlp>=2024.8"
# Short clip of a horse in a riding arena (CC-BY — verify before any commercial use).
!yt-dlp -f 'best[ext=mp4][height<=720]' --download-sections '*0:00-0:30' -o /content/sample_horse.mp4 'https://www.youtube.com/watch?v=Dy9Tcm4S22Q'
import os
print('Sample:', '/content/sample_horse.mp4', f'({os.path.getsize("/content/sample_horse.mp4") / 1e6:.1f} MB)')

## 3. DLC SuperAnimal-Quadruped inference (zero-shot)

In [ ]:
import os, deeplabcut
os.makedirs('/content/outputs', exist_ok=True)

deeplabcut.video_inference_superanimal(
    videos=['/content/sample_horse.mp4'],
    superanimal_name='superanimal_quadruped',
    model_name='hrnet_w32',
    detector_name='fasterrcnn_resnet50_fpn_v2',
    video_adapt=False,
    dest_folder='/content/outputs',
    pseudo_threshold=0.1,
)

## 4. Visual inspection

In [ ]:
import cv2, glob, numpy as np, matplotlib.pyplot as plt

annotated = sorted(glob.glob('/content/outputs/*labeled*.mp4'))
assert annotated, 'Nie wygenerowano annotated video — sprawdz logi cell 5.'
vid = annotated[0]
print('Annotated:', vid)

cap = cv2.VideoCapture(vid)
n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
idx = np.linspace(0, n - 1, 5, dtype=int)
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, i in zip(axes, idx):
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(i))
    ok, frame = cap.read()
    if ok:
        ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    ax.set_title(f'klatka {i}')
    ax.axis('off')
cap.release()
plt.tight_layout()
plt.savefig('/content/outputs/sample_keypoints_grid.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Download the results to your laptop

In the Files panel (left column), right-click on `outputs/` → Download. Copy the annotated MP4 and grid PNG into `poc/outputs/` on your laptop and record the results in `GATE.md`.

**Note:** this notebook does not replace `01_read_my_ears_replicate.ipynb`. GATE.md item 3 (Read My Ears) requires a separate run — you can do it in a new Colab notebook, but for Phase 0 the DLC sanity check + checking off items 1 and 2 of GATE is enough.
